# California Housing — Doğrusal Regresyon ile Konut Değeri Tahmini
**Yüksek lisans vize çalışması | Tek model, tek notebook**

**Araştırma sorusu:** Sekiz sayısal bölge özelliği kullanılarak California'daki bir bölgenin medyan konut değeri tahmin edilebilir mi?

**Yöntem:** Çok değişkenli **Linear Regression (OLS)**. LSTM, TensorFlow, zaman serisi, alternatif modeller ve karmaşık eğitim döngüleri **yoktur**.

**Çalıştırma:** Jupyter'da `Run All` seçin. İlk çalıştırmada internet gerekir; veri bir kez indirilip `california_housing.csv` dosyasına kaydedilir. Grafikler notebook içinde görünür. İsterseniz son bölümde sonuçlar ayrıca dosyaya yazılır.

**Kurulum (terminal):** `python -m pip install notebook pandas numpy matplotlib scikit-learn` ardından `jupyter notebook`.

## 1. Kütüphaneler
- **pandas:** tablo işlemleri
- **numpy:** sayısal hesaplamalar
- **matplotlib:** grafikler
- **scikit-learn:** veri seti, eğitim/test bölme, LinearRegression ve metrikler

Burada yalnız gerekli kütüphaneleri içeri aktarıyoruz.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error, explained_variance_score, max_error

RANDOM_STATE = 42  # Aynı train/test bölmesini yeniden üretmek için.
CSV_PATH = Path("california_housing.csv")


## 2. Veri setini indir, kaydet ve aç
California Housing: **20.640 bölge**, **8 sayısal giriş**, **1 sayısal hedef**. Bir satır tek bir ev değil, bir **nüfus sayımı bölgesi** hakkındaki toplu istatistiklerdir. Hedef `MedHouseVal`, bölgenin medyan konut değeridir ve **100.000 ABD doları** birimindedir: örneğin `2.5` yaklaşık **250.000 USD** demektir. Veriler tarihsel olup bugünün piyasa fiyatlarını temsil etmez.

CSV varsa yeniden indirilmez. Yoksa scikit-learn veriyi internetten indirir, sonra CSV kaydedilir.

In [ ]:
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    print("Yerel CSV dosyası açıldı:", CSV_PATH.resolve())
else:
    housing = fetch_california_housing(as_frame=True)
    df = housing.frame.copy()
    df.to_csv(CSV_PATH, index=False)
    print("Veri indirildi ve CSV olarak kaydedildi:", CSV_PATH.resolve())

print("Satır ve sütun sayısı:", df.shape)
display(df.head(10))


## 3. Sütunları tanı
| Sütun | Tanım |
|---|---|
| `MedInc` | Bölgenin medyan hane geliri (veri setine özgü ölçek) |
| `HouseAge` | Bölgedeki evlerin medyan yaşı |
| `AveRooms` | Hane başına ortalama oda sayısı |
| `AveBedrms` | Hane başına ortalama yatak odası sayısı |
| `Population` | Bölgenin nüfusu |
| `AveOccup` | Hane başına ortalama kişi sayısı |
| `Latitude` | Enlem |
| `Longitude` | Boylam |
| `MedHouseVal` | **Hedef:** medyan konut değeri (100.000 USD biriminde) |

**Özellik (X):** Tahmin için kullanılan sütunlar. **Hedef (y):** Tahmin edilen sütun. `MedHouseVal` girişlere dahil edilmez.

In [ ]:
print("Sütunlar:", list(df.columns))
print("Eksik değer sayısı:")
display(df.isna().sum().to_frame("Eksik"))
print("Özet istatistikler:")
display(df.describe().round(2))


## 4. Veriyi incele
Dağılım grafiği, hedef değerlerin hangi aralıklarda yoğunlaştığını gösterir. Saçılım grafiği ise medyan gelir ile konut değeri arasındaki ilişkiyi görselleştirir. **İlişki nedensellik kanıtı değildir.** Veri setinde yüksek konut değerleri üst sınırda kırpılmıştır; bu durum tahminleri etkileyebilir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["MedHouseVal"] * 100_000, bins=35, edgecolor="white")
axes[0].set(title="Medyan konut değeri dağılımı", xlabel="USD", ylabel="Bölge sayısı")
axes[1].scatter(df["MedInc"], df["MedHouseVal"] * 100_000, s=5, alpha=0.25)
axes[1].set(title="Gelir ve konut değeri", xlabel="MedInc", ylabel="Medyan konut değeri (USD)")
plt.tight_layout()
plt.show()


## 5. Giriş ve hedefi ayır; eğitim/test kümelerini oluştur
**Eğitim (%80):** Model katsayılarını öğrenir. **Test (%20):** Modelin görmediği örneklerde performansını ölçer. `random_state=42` aynı rastgele bölmeyi tekrar üretir.

**Aykırı değer temizliği (satır silmeden):** Bazı ölçümlerde, özellikle hane başına kişi sayısında (`AveOccup`), çok uç değerler bulunur. Bunların etkisini azaltmak için belirli dört sütunu eğitim verisindeki **%1 ve %99 yüzdelik sınırlarına** kırpıyoruz (winsorization). Sınırları **yalnızca eğitim verisinden** öğreniyoruz; aynı sınırları test girdilerine de uyguluyoruz. Böylece test hedeflerine dokunmadan aşırı uçların tahminleri bozmasını azaltıyoruz. Hiçbir satırı silmiyoruz.

**Not:** Bu dönüşüm veri temizliğidir; ikinci bir regresyon modeli değildir. Konum (`Latitude`, `Longitude`) sütunlarını kırpmıyoruz. Basit OLS için `StandardScaler` zorunlu değildir; katsayılar dönüştürülmüş özelliklerin kendi birimleri üzerinden yorumlanır.

In [ ]:
X = df.drop(columns="MedHouseVal")  # Sekiz giriş özelliği
y = df["MedHouseVal"]               # Tahmin edilecek konut değeri

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

# Aykırı değer temizliği: satır silmeden uç değerleri sınırla.
# Sınırları yalnızca eğitim verisi belirler (veri sızıntısı yok).
outlier_columns = ["AveRooms", "AveBedrms", "Population", "AveOccup"]
lower_bounds = X_train[outlier_columns].quantile(0.01)
upper_bounds = X_train[outlier_columns].quantile(0.99)

X_train = X_train.copy()
X_test = X_test.copy()
for column in outlier_columns:
    X_train[column] = X_train[column].clip(lower_bounds[column], upper_bounds[column])
    X_test[column] = X_test[column].clip(lower_bounds[column], upper_bounds[column])

print("Aykırı değer temizliği uygulandı (yüzdelik %1–%99).")
print("Silinen satır sayısı: 0")
print(f"Eğitim: {len(X_train):,} satır | Test: {len(X_test):,} satır")
print("Giriş boyutu:", X_train.shape, "| Hedef boyutu:", y_train.shape)


## 6. Tek model: Linear Regression
**Doğrusal regresyon**, girdilerin ağırlıklı toplamıyla sayısal hedefi tahmin eder:

\[\hat{y}=b+w_1x_1+w_2x_2+\cdots+w_8x_8\]

- `w`: Her özelliğin katsayısı
- `b`: Sabit terim (intercept)
- `fit`: Eğitim verisine göre katsayıları öğrenir
- `predict`: Öğrenilen katsayılarla test verisini tahmin eder

**Dikkat:** Katsayıların büyüklüğü doğrudan önem sıralaması değildir; katsayılar nedensellik kanıtlamaz. Dört özellikteki aşırı değerler eğitim kaynaklı sınırlarla kırpıldığından bu özelliklerin etkileri kırpılmış ölçekte yorumlanır.

In [ ]:
model = LinearRegression()      # Tek regresyon modelini oluştur.
model.fit(X_train, y_train)    # Katsayıları yalnız eğitim verisiyle öğren.
y_pred = model.predict(X_test) # Görülmemiş test verisi için tahmin üret.

coefficients = pd.DataFrame({"Özellik": X.columns, "Katsayı": model.coef_})
print("Sabit terim (intercept):", round(float(model.intercept_), 5))
display(coefficients.round(5))


## 7. Test metrikleri — hepsi aynı model için
**MAE:** Mutlak hataların ortalaması; küçük olması iyi. **MSE:** Hataların karelerinin ortalaması; büyük hataları daha çok cezalandırır. **RMSE:** MSE'nin karekökü; hedefle aynı birimdedir. **MedAE:** Mutlak hataların ortancası. **Max Error:** En büyük mutlak hata. **R²:** Sabit test ortalaması tahminine göre karesel hata başarısı; 1 en iyi, 0 referans düzeyi, negatif de olabilir. **Explained Variance:** Açıklanan varyans skoru.

**R² bir isabet yüzdesi değildir.** MAE/RMSE/MedAE/Max Error, burada kolay yorumlanması için USD'ye çevrilir; MSE ise USD² cinsindedir. Veri setinin hedefi bölgesel medyan değeridir, tekil ev satış fiyatı değildir.

In [ ]:
# Hedef 100.000 USD biriminde tutulduğu için değerlendirmede USD'ye çeviriyoruz.
y_true_usd = y_test.to_numpy() * 100_000
y_pred_usd = y_pred * 100_000
errors_usd = y_true_usd - y_pred_usd

results = {
    "MAE (USD)": mean_absolute_error(y_true_usd, y_pred_usd),
    "MSE (USD²)": mean_squared_error(y_true_usd, y_pred_usd),
    "RMSE (USD)": np.sqrt(mean_squared_error(y_true_usd, y_pred_usd)),
    "Median AE (USD)": median_absolute_error(y_true_usd, y_pred_usd),
    "Max Error (USD)": max_error(y_true_usd, y_pred_usd),
    "R²": r2_score(y_true_usd, y_pred_usd),
    "Explained Variance": explained_variance_score(y_true_usd, y_pred_usd),
}
display(pd.DataFrame({"Metrik": list(results), "Test sonucu": list(results.values())}).round(4))
print(f"Ortalama mutlak hata: ${results['MAE (USD)']:,.0f}")
print(f"R²: {results['R²']:.4f} (doğruluk yüzdesi DEĞİLDİR)")


## 8. Gerçek ve tahmin grafikleri
**Sol:** İdeal durumda noktalar kesikli diyagonal üzerinde olur. **Sağ:** Artık (residual) = gerçek − tahmin. Sıfır çevresinde dağılmaları istenir. Sistematik desenler doğrusal modelin sınırlamalarını gösterebilir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_true_usd, y_pred_usd, s=8, alpha=0.3)
lo = min(y_true_usd.min(), y_pred_usd.min())
hi = max(y_true_usd.max(), y_pred_usd.max())
axes[0].plot([lo, hi], [lo, hi], linestyle="--", color="black")
axes[0].set(title="Gerçek ve tahmin", xlabel="Gerçek (USD)", ylabel="Tahmin (USD)")
axes[1].scatter(y_pred_usd, errors_usd, s=8, alpha=0.3)
axes[1].axhline(0, linestyle="--", color="black")
axes[1].set(title="Artık (hata) grafiği", xlabel="Tahmin (USD)", ylabel="Gerçek − tahmin (USD)")
plt.tight_layout()
plt.show()

comparison = pd.DataFrame({"Gerçek (USD)": y_true_usd[:10], "Tahmin (USD)": y_pred_usd[:10],
                           "Mutlak hata (USD)": np.abs(errors_usd[:10])})
display(comparison.round(0))


## 9. Sonuçları kaydet (isteğe bağlı)
Veri `california_housing.csv` olarak zaten kaydedildi. Bu hücre ayrıca test metriklerini JSON, test tahminlerini CSV olarak kaydeder. **Eğitim, tahmin ve grafikler yine yalnız bu notebook içindedir.**

In [ ]:
Path("linear_regression_metrics.json").write_text(
    json.dumps({k: float(v) for k, v in results.items()}, indent=2, ensure_ascii=False), encoding="utf-8"
)
pd.DataFrame({"actual_usd": y_true_usd, "predicted_usd": y_pred_usd,
              "error_usd": errors_usd}).to_csv("linear_regression_predictions.csv", index=False)
print("Kaydedildi: linear_regression_metrics.json ve linear_regression_predictions.csv")


## 10. Hocaya sunum için kısa özet
**Problem:** California'daki nüfus sayımı bölgelerinin medyan konut değerini sekiz sayısal değişkenden tahmin etmek.

**Veri:** California Housing, 20.640 gözlem, 8 girdi, 1 sürekli hedef.

**Yöntem:** Yalnız **çok değişkenli doğrusal regresyon (OLS)**.

**Deney:** %80 eğitim / %20 test. Aykırı giriş ölçümlerinin etkisini azaltmak için dört sütunda eğitim verisinden hesaplanan %1–%99 sınırlarıyla kırpma yapıldı; satır silinmedi. Aynı dönüşüm test girdilerine uygulandı.

**Değerlendirme:** MAE, MSE, RMSE, MedAE, Max Error, R², explained variance; gerçek-tahmin ve artık grafikleri.

**Sınırlılıklar:** Veri tarihsel ve bölge düzeyindedir. Hedefte üst sınır etkisi bulunur. Doğrusal model her ilişkiyi açıklayamaz. Test başarısı garanti değildir; gerçek sonuçlar Notebook çalıştırılınca hesaplanır.

**Kaynak:** [scikit-learn California Housing resmi dokümantasyonu](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html).